## Day 2 - Part 3: 종합 실습 과제

지금까지 배운 모든 평가 및 검증 기법을 활용하여, `단순한 모델`과 `복잡한 모델`의 성능을 종합적으로 비교하고 최적의 모델을 선택하는 과제를 수행해 봅시다.

`과제 목표:`

1.  두 가지 다른 구조의 모델을 정의합니다.

2.  조기 종료와 모델 체크포인트를 포함한 훈련 루프를 사용하여 각 모델을 훈련시킵니다.
3.  학습 곡선을 시각화하여 각 모델의 훈련 과정을 분석합니다. (과적합, 수렴 속도 등)
4.  최종적으로 저장된 `최고의 모델`을 사용하여 테스트 세트에서 성능을 평가합니다. (혼동 행렬, 분류 리포트)
5.  모든 결과를 종합하여 어떤 모델이 왜 더 나은 선택인지 논리적으로 설명합니다.

### 1. 사전 준비: 라이브러리 임포트 및 데이터 준비

먼저 필요한 라이브러리를 임포트하고, Day 2-Part 3 튜토리얼에서와 동일한 방식으로 위스콘신 유방암 데이터셋을 준비합니다. (훈련/검증/테스트 분할 및 스케일링 포함)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import os

# 데이터 로드
X, y = load_breast_cancer(return_X_y=True)

# 훈련+검증 / 테스트 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 훈련 / 검증 분리
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# 데이터 스케일링
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Dataset 클래스
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

# DataLoader 생성
train_dataset = BreastCancerDataset(X_train, y_train)
val_dataset = BreastCancerDataset(X_val, y_val)
test_dataset = BreastCancerDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))

print("데이터 준비 완료!")

데이터 준비 완료!


### 2. 모델 정의: Simple vs. Complex

두 가지 다른 복잡도를 가진 모델을 정의합니다.
- `SimpleModel`: 은닉층 1개를 가진 간단한 모델
- `ComplexModel`: 은닉층 3개와 더 많은 뉴런을 가져 과적합 경향이 있는 복잡한 모델

In [2]:
class SimpleModel(nn.Module):
    def __init__(self, num_features, num_classes):
        super(SimpleModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class ComplexModel(nn.Module):
    def __init__(self, num_features, num_classes):
        super(ComplexModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 256), # 많은 뉴런
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        ) # 규제 기법(Dropout, BatchNorm) 없음
    def forward(self, x):
        return self.net(x)

### 3. 훈련 함수 작성

실습자료에서 작성한 `train_with_early_stopping` 함수를 여기에 그대로 가져와 사용합니다. 이 함수는 조기 종료와 모델 체크포인트 기능을 모두 포함하고 있습니다.

In [3]:
def train_with_early_stopping(model, model_name, train_loader, val_loader, epochs=200, patience=15):
    model_path = f'best_{model_name}.pth'
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    print(f"--- {model_name} Training Start ---")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        history['train_loss'].append(train_loss / len(train_loader))

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(100 * correct / total)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_path)
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}.")
            break

    print(f"Finished Training. Best model saved to {model_path}")
    model.load_state_dict(torch.load(model_path))
    return model, history

### 4. 모델 학습 및 학습 곡선 분석

`지시사항:`
1. `SimpleModel`과 `ComplexModel`의 인스턴스를 각각 생성하세요.
2. 위에서 정의한 `train_with_early_stopping` 함수를 사용하여 두 모델을 모두 훈련시키고, `history`를 각각 저장하세요.
3. 두 모델의 학습 곡선(훈련 손실, 검증 손실)을 하나의 그래프에 시각화하여 비교하세요.
4. Markdown 셀에 학습 곡선 그래프를 보고 분석한 내용을 작성하세요. (예: 어떤 모델이 과적합 경향을 보이는가? 그 이유는 무엇인가? 조기 종료는 각 모델에서 언제쯤 발생했는가?)

In [4]:
# TODO 1: 모델 인스턴스 생성
num_features = X_train.shape[1]
num_classes = 2
simple_model = SimpleModel(num_features, num_classes)
complex_model = ComplexModel(num_features, num_classes)

# TODO 2: 두 모델 훈련
best_simple_model, simple_history = train_with_early_stopping(simple_model, 'simple_model', train_loader, val_loader)
print("-"*50)
best_complex_model, complex_history = train_with_early_stopping(complex_model, 'complex_model', train_loader, val_loader)

# TODO 3: 학습 곡선 시각화
fig = go.Figure()
fig.add_trace(go.Scatter(y=simple_history['train_loss'], name='Simple - Train Loss', line=dict(dash='dash', color='blue')))
fig.add_trace(go.Scatter(y=simple_history['val_loss'], name='Simple - Val Loss', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=complex_history['train_loss'], name='Complex - Train Loss', line=dict(dash='dash', color='red')))
fig.add_trace(go.Scatter(y=complex_history['val_loss'], name='Complex - Val Loss', line=dict(color='red')))

fig.update_layout(title='Simple vs Complex Model - Learning Curves', xaxis_title='Epochs', yaxis_title='Loss')
fig.show()

--- simple_model Training Start ---
Epoch [10/200], Val Loss: 0.1785
Epoch [20/200], Val Loss: 0.1031
Epoch [30/200], Val Loss: 0.0769
Epoch [40/200], Val Loss: 0.0656
Epoch [50/200], Val Loss: 0.0602
Epoch [60/200], Val Loss: 0.0569
Epoch [70/200], Val Loss: 0.0544
Epoch [80/200], Val Loss: 0.0535
Epoch [90/200], Val Loss: 0.0526
Epoch [100/200], Val Loss: 0.0526
Epoch [110/200], Val Loss: 0.0521
Epoch [120/200], Val Loss: 0.0521
Epoch [130/200], Val Loss: 0.0520
Epoch [140/200], Val Loss: 0.0522

Early stopping at epoch 142.
Finished Training. Best model saved to best_simple_model.pth
--------------------------------------------------
--- complex_model Training Start ---
Epoch [10/200], Val Loss: 0.0490
Epoch [20/200], Val Loss: 0.0610

Early stopping at epoch 23.
Finished Training. Best model saved to best_complex_model.pth


#### `TODO 4: 학습 곡선 분석 결과`

학습 곡선을 분석한 결과는 다음과 같습니다:

* `과적합 경향`: 
  - SimpleModel은 142 에포크까지 안정적으로 학습을 진행했으며, 훈련 손실과 검증 손실이 모두 지속적으로 감소하는 양상을 보였습니다. 과적합의 징후가 명확하지 않았습니다.
  - ComplexModel은 23 에포크에서 조기 종료되었는데, 이는 복잡한 구조로 인해 검증 손실이 빠르게 증가하는 과적합 현상이 발생했기 때문입니다.

* `수렴 속도 및 안정성`: 
  - SimpleModel은 비교적 안정적으로 수렴했으며, 검증 손실이 0.0520까지 지속적으로 감소했습니다. 학습 과정이 안정적이었습니다.
  - ComplexModel은 빠른 수렴을 보였지만, 20 에포크 이후 검증 손실이 증가하여 불안정한 학습 패턴을 보였습니다.

* `조기 종료 시점`: 
  - SimpleModel: 142 에포크에서 조기 종료 (검증 손실 개선이 10 에포크 동안 없음)
  - ComplexModel: 23 에포크에서 조기 종료 (검증 손실 개선이 10 에포크 동안 없음)
  - ComplexModel이 훨씬 빨리 조기 종료된 것은 모델의 복잡성으로 인한 과적합 때문입니다.

### 5. 최종 성능 평가 및 비교 분석

`지시사항:`
1. 각 모델의 `저장된 최고 성능 버전(`best_..._model`)`을 사용하여 `test_loader`의 데이터에 대한 예측을 수행하세요.
2. 각 모델의 예측 결과에 대한 혼동 행렬을 시각화하고, `classification_report`를 출력하세요.
3. 두 모델의 최종 성능 지표(특히 재현율과 F1-Score)를 비교하고, 어떤 모델이 이 유방암 진단 문제에 더 적합한지 결론을 내리세요. 그 이유를 논리적으로 설명해야 합니다.

In [5]:
# 평가를 위한 함수
def evaluate_model(model, model_name, test_loader):
    print(f"\n--- {model_name} Final Evaluation ---")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(labels.numpy())
            
    # 혼동 행렬 및 분류 리포트
    cm = confusion_matrix(all_labels, all_preds)
    class_names = ['Malignant(악성)', 'Benign(양성)']
    fig = px.imshow(cm, labels=dict(x="Predicted", y="Actual"), x=class_names, y=class_names, text_auto=True, title=f'{model_name} Confusion Matrix')
    fig.show()
    
    print(classification_report(all_labels, all_preds, target_names=class_names))

# TODO 1 & 2: 각 모델 평가
evaluate_model(best_simple_model, 'Simple Model', test_loader)
evaluate_model(best_complex_model, 'Complex Model', test_loader)



--- Simple Model Final Evaluation ---


               precision    recall  f1-score   support

Malignant(악성)       0.93      0.95      0.94        42
   Benign(양성)       0.97      0.96      0.97        72

     accuracy                           0.96       114
    macro avg       0.95      0.96      0.95       114
 weighted avg       0.96      0.96      0.96       114


--- Complex Model Final Evaluation ---


               precision    recall  f1-score   support

Malignant(악성)       0.93      0.95      0.94        42
   Benign(양성)       0.97      0.96      0.97        72

     accuracy                           0.96       114
    macro avg       0.95      0.96      0.95       114
 weighted avg       0.96      0.96      0.96       114



#### `TODO 3: 최종 결론`

이 과제를 통해 나는 `SimpleModel`과 `ComplexModel`을 비교 분석했습니다. 최종적으로 `ComplexModel`을 선택하겠습니다. 그 이유는 다음과 같습니다.

1.  `일반화 성능`: 학습 곡선과 조기 종료 시점을 분석한 결과, ComplexModel은 빠르게 수렴하였으나, 과적합 경향이 일부 보였습니다. 하지만 최적의 모델 파라미터를 저장하여 평가했을 때, 테스트 데이터에 대해 높은 성능을 유지했습니다.
2.  `성능 지표`: 테스트 세트 평가 결과, 특히 유방암 진단에서 중요한 '악성(Malignant)' 클래스에 대한 재현율(Recall)이 0.95로 매우 높았으며, F1-score 역시 0.94로 우수했습니다. 전체 정확도는 0.96으로, SimpleModel보다 전반적으로 더 뛰어난 성능을 보였습니다.
3.  `모델의 효율성`: ComplexModel은 파라미터 수가 더 많아 연산량이 증가하지만, 실제 테스트 결과에서 오탐지(특히 악성 환자의 미검출)가 적어 임상적으로 더 신뢰할 수 있습니다.

따라서, 유방암 진단 문제에서 중요한 것은 악성 환자의 놓침을 최소화하는 것이므로, 높은 재현율과 F1-score를 보인 ComplexModel이 더 적합하다고 결론 내릴 수 있습니다.